# FNI-LLM Training Notebook v3
## Cameroon Language Model — Self-Contained Training

**All definitions are in Cell 2. Just run Cell 1 then Cell 2 — that's it.**


In [ ]:
# ============================================================
# CELL 1: Mount Drive + Sync Code
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/FNI_AI_LLM
!git pull origin master

import sys
sys.path.insert(0, '/content/drive/MyDrive/FNI_AI_LLM')
print('Ready!')

In [ ]:
# ============================================================
# CELL 2: FULL SETUP — Imports, Classes, Data, Model
# Run this after Cell 1. Everything is defined here.
# ============================================================

# --- Imports ---
import re, os, json, time
import numpy as np
from collections import Counter
import torch
import torch.nn as nn

print(f'PyTorch: {torch.__version__}')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# --- CONFIG ---
LANGUAGE   = 'english'   # Change to: 'french', 'bayangi', 'douala'
SEQ_LEN    = 64
MAX_VOCAB  = 100000
BATCH_SIZE = 64
EPOCHS     = 30
LR         = 3e-4
WARMUP     = 3
DATA_PATH  = f'data/cameroon_languages/{LANGUAGE}/processed/{LANGUAGE}_clean.txt'

print(f'Language: {LANGUAGE}')
print(f'Config: seq={SEQ_LEN}, vocab={MAX_VOCAB}, epochs={EPOCHS}')

In [ ]:
# ============================================================
# CELL 3: Tokenizer
# ============================================================

class ImprovedTokenizer:
    PAD, UNK, START, END = '<PAD>', '<UNK>', '<START>', '<END>'

    def __init__(self, max_vocab=100000):
        self.max_vocab   = max_vocab
        self.word_to_id  = {}
        self.id_to_word  = {}
        self.vocab_size  = 0

    def tokenize(self, text):
        return re.findall(r"[\w']+|[.,!?;]", text.lower())

    def build_vocab(self, corpus):
        freq  = Counter(self.tokenize(corpus))
        words = [w for w, _ in freq.most_common(self.max_vocab - 4)]
        vocab = [self.PAD, self.UNK, self.START, self.END] + words
        self.word_to_id = {w: i for i, w in enumerate(vocab)}
        self.id_to_word = {i: w for w, i in self.word_to_id.items()}
        self.vocab_size = len(vocab)
        unk_count = sum(c for w, c in freq.items()
                        if w not in self.word_to_id)
        total     = sum(freq.values())
        print(f'Vocab size: {self.vocab_size:,}')
        print(f'UNK rate:   {unk_count/total:.1%}')

    def encode(self, text):
        unk = self.word_to_id[self.UNK]
        return [self.word_to_id.get(w, unk)
                for w in self.tokenize(text)]

    def decode(self, ids):
        skip  = {self.PAD, self.UNK, self.START, self.END}
        words = [self.id_to_word.get(i, '') for i in ids]
        return ' '.join(w for w in words if w and w not in skip)

    def save(self, path):
        with open(path, 'w', encoding='utf-8') as f:
            json.dump({'word_to_id': self.word_to_id,
                       'vocab_size': self.vocab_size}, f)

    def load(self, path):
        with open(path, encoding='utf-8') as f:
            data = json.load(f)
        self.word_to_id = data['word_to_id']
        self.id_to_word = {int(i): w
                           for w, i in self.word_to_id.items()}
        self.vocab_size = data['vocab_size']

print('ImprovedTokenizer defined!')

In [ ]:
# ============================================================
# CELL 4: Transformer Model
# ============================================================

class FNITransformer(nn.Module):
    def __init__(self, vocab_size, d_model=256, num_heads=8,
                 d_ff=1024, num_layers=4,
                 max_seq_len=64, dropout=0.1):
        super().__init__()
        self.vocab_size   = vocab_size
        self.d_model      = d_model
        self.embedding    = nn.Embedding(vocab_size, d_model,
                                         padding_idx=0)
        self.pos_emb      = nn.Embedding(max_seq_len, d_model)
        self.dropout      = nn.Dropout(dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=num_heads,
            dim_feedforward=d_ff, dropout=dropout,
            batch_first=True, norm_first=True
        )
        self.transformer  = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers)
        self.norm         = nn.LayerNorm(d_model)
        self.output       = nn.Linear(d_model, vocab_size, bias=False)
        self.output.weight = self.embedding.weight  # weight tying
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, x):
        B, T    = x.shape
        pos     = torch.arange(T, device=x.device).unsqueeze(0)
        x       = self.dropout(self.embedding(x) + self.pos_emb(pos))
        x       = self.transformer(x)
        x       = self.norm(x)
        return self.output(x)

    def count_params(self):
        return sum(p.numel() for p in self.parameters())

print('FNITransformer defined!')

In [ ]:
# ============================================================
# CELL 5: Load Data
# ============================================================

with open(DATA_PATH, encoding='utf-8') as f:
    corpus = f.read()

print(f'Corpus size: {len(corpus):,} characters')
print(f'Lines: {corpus.count(chr(10)):,}')

tokenizer = ImprovedTokenizer(max_vocab=MAX_VOCAB)
tokenizer.build_vocab(corpus)

# Save tokenizer
os.makedirs(f'models/{LANGUAGE}', exist_ok=True)
tokenizer.save(f'models/{LANGUAGE}/tokenizer.json')
print(f'Tokenizer saved!')

# Encode corpus
all_ids = tokenizer.encode(corpus)
print(f'Total tokens: {len(all_ids):,}')
print(f'UNK tokens:   {all_ids.count(1):,} ({all_ids.count(1)/len(all_ids):.1%})')

# Build samples
samples = []
for i in range(0, len(all_ids) - SEQ_LEN, SEQ_LEN // 2):
    chunk = all_ids[i: i + SEQ_LEN + 1]
    if len(chunk) == SEQ_LEN + 1:
        samples.append(chunk)

np.random.seed(42)
np.random.shuffle(samples)
n             = len(samples)
train_samples = samples[:int(n * 0.8)]
val_samples   = samples[int(n * 0.8): int(n * 0.9)]

print(f'Train: {len(train_samples):,} | Val: {len(val_samples):,}')

In [ ]:
# ============================================================
# CELL 6: Build Model
# ============================================================

model = FNITransformer(
    vocab_size  = tokenizer.vocab_size,
    d_model     = 256,
    num_heads   = 8,
    d_ff        = 1024,
    num_layers  = 4,
    max_seq_len = SEQ_LEN,
    dropout     = 0.1
).to(DEVICE)

print(f'Parameters: {model.count_params():,}')
print(f'Vocab size: {tokenizer.vocab_size:,}')
print(f'Device:     {DEVICE}')

In [ ]:
# ============================================================
# CELL 7: Training Loop
# ============================================================

def make_batches(samples, batch_size, shuffle=True):
    idx = np.arange(len(samples))
    if shuffle:
        np.random.shuffle(idx)
    for i in range(0, len(idx) - batch_size, batch_size):
        batch = [samples[j] for j in idx[i: i + batch_size]]
        arr   = np.array(batch)
        yield arr[:, :-1], arr[:, 1:]

def get_lr(epoch):
    if epoch < WARMUP:
        return LR * (epoch + 1) / WARMUP
    progress = (epoch - WARMUP) / max(EPOCHS - WARMUP, 1)
    return LR * 0.5 * (1 + np.cos(np.pi * progress))

optimizer     = torch.optim.AdamW(model.parameters(),
                                   lr=LR, weight_decay=0.01)
criterion     = nn.CrossEntropyLoss(ignore_index=0)
best_val_loss = float('inf')
log           = []

os.makedirs('models/checkpoints', exist_ok=True)
os.makedirs('logs', exist_ok=True)

print(f'Training {LANGUAGE} for {EPOCHS} epochs...\n')

for epoch in range(1, EPOCHS + 1):
    for g in optimizer.param_groups:
        g['lr'] = get_lr(epoch)

    # Train
    model.train()
    t_losses = []
    t0 = time.time()
    for X_np, y_np in make_batches(train_samples, BATCH_SIZE):
        X    = torch.tensor(X_np, dtype=torch.long).to(DEVICE)
        y    = torch.tensor(y_np, dtype=torch.long).to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(
            model(X).reshape(-1, tokenizer.vocab_size),
            y.reshape(-1)
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        t_losses.append(loss.item())

    # Validate
    model.eval()
    v_losses = []
    with torch.no_grad():
        for X_np, y_np in make_batches(val_samples, BATCH_SIZE,
                                        shuffle=False):
            X    = torch.tensor(X_np, dtype=torch.long).to(DEVICE)
            y    = torch.tensor(y_np, dtype=torch.long).to(DEVICE)
            loss = criterion(
                model(X).reshape(-1, tokenizer.vocab_size),
                y.reshape(-1)
            )
            v_losses.append(loss.item())

    tl      = float(np.mean(t_losses))
    vl      = float(np.mean(v_losses))
    elapsed = time.time() - t0

    # Save best checkpoint
    if vl < best_val_loss:
        best_val_loss = vl
        torch.save({
            'model':  model.state_dict(),
            'config': {
                'vocab_size':  tokenizer.vocab_size,
                'd_model':     256,
                'num_heads':   8,
                'd_ff':        1024,
                'num_layers':  4,
                'max_seq_len': SEQ_LEN
            }
        }, f'models/checkpoints/{LANGUAGE}_best.pt')

    log.append({
        'epoch':      epoch,
        'train_loss': round(tl, 4),
        'val_loss':   round(vl, 4),
        'time':       round(elapsed, 2)
    })

    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{EPOCHS} | '
              f'train={tl:.4f} | val={vl:.4f} | '
              f'lr={get_lr(epoch):.2e} | {elapsed:.1f}s')

with open(f'logs/{LANGUAGE}_v3_training.json', 'w') as f:
    json.dump(log, f, indent=2)

print(f'\nBest val loss: {best_val_loss:.4f}')
print(f'Saved: models/checkpoints/{LANGUAGE}_best.pt')

In [ ]:
# ============================================================
# CELL 8: Plot Training Curve
# ============================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

tl = [e['train_loss'] for e in log]
vl = [e['val_loss']   for e in log]

plt.figure(figsize=(10, 4))
plt.plot(tl, label='Train Loss')
plt.plot(vl, label='Val Loss')
plt.title(f'FNI-LLM Training - {LANGUAGE.capitalize()}')
plt.xlabel('Epoch')
plt.ylabel('Cross-Entropy Loss')
plt.legend()
plt.grid(True)
plt.tight_layout()
os.makedirs('docs/visualizations', exist_ok=True)
plt.savefig(f'docs/visualizations/{LANGUAGE}_v3_curve.png', dpi=100)
plt.close()
print(f'Saved: docs/visualizations/{LANGUAGE}_v3_curve.png')

In [ ]:
# ============================================================
# CELL 9: Generate Text
# ============================================================

def generate(model, tokenizer, prompt,
             max_new_tokens=25, temperature=0.7,
             top_k=20, device='cpu'):
    model.eval()
    ids  = tokenizer.encode(prompt)
    seen = {}
    with torch.no_grad():
        for _ in range(max_new_tokens):
            x      = torch.tensor(
                [ids[-SEQ_LEN:]], dtype=torch.long).to(device)
            logits = model(x)[0, -1].cpu().numpy().astype(float)

            # Block PAD and UNK
            logits[0] = -1e9
            logits[1] = -1e9

            # Repetition penalty
            for tid, cnt in seen.items():
                logits[tid] -= 3.0 * cnt

            # Top-k sampling
            logits      = logits / temperature
            top_k_idx   = np.argsort(logits)[-top_k:]
            top_logits  = logits[top_k_idx]
            probs       = np.exp(top_logits - np.max(top_logits))
            probs       = probs / probs.sum()
            next_id     = int(np.random.choice(top_k_idx, p=probs))

            seen[next_id] = seen.get(next_id, 0) + 1
            ids.append(next_id)

    # Return only newly generated words
    prompt_len = len(tokenizer.encode(prompt))
    return tokenizer.decode(ids[prompt_len:])

# Load best checkpoint
ckpt = torch.load(f'models/checkpoints/{LANGUAGE}_best.pt',
                   map_location=DEVICE)
model.load_state_dict(ckpt['model'])
print(f'Loaded best checkpoint\n')

prompts = [
    'cameroon is a country in',
    'the people of cameroon are',
    'douala is the largest city',
    'the official languages of cameroon are',
    'mount cameroon is the highest',
    'the history of cameroon',
]

print(f'=== GENERATED TEXT ({LANGUAGE.upper()}) ===')
for p in prompts:
    out = generate(model, tokenizer, p,
                   max_new_tokens=20, temperature=0.7,
                   top_k=20, device=DEVICE)
    print(f'Prompt : "{p}"')
    print(f'Output : "{out}"')
    print()

In [ ]:
# ============================================================
# CELL 10: Push to GitHub
# ============================================================
!git config --global user.email 'coursdenatationcmr@gmail.com'
!git config --global user.name 'Ronald'
%cd /content/drive/MyDrive/FNI_AI_LLM
!git add logs/ docs/visualizations/ models/{LANGUAGE}/
!git commit -m '[Year3] {LANGUAGE} v3 - 100K vocab, improved generation'
!git push origin master
print('Pushed to GitHub!')